In [1]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth, association_rules
import warnings
warnings.filterwarnings("ignore")


In [3]:
df = pd.read_csv("retail_cleaned.csv",low_memory=False)
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [4]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df = df[df['Quantity'] > 0]
df.dropna(subset=['Description', 'CustomerID'], inplace=True)
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']


In [7]:
basket = (
    df
    .groupby(['InvoiceNo', 'Description'])['Quantity']
    .sum()
    .unstack()
    .fillna(0)
)


In [8]:
basket = basket > 0


In [9]:
frequent_itemsets = fpgrowth(basket, min_support=0.01, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets.head()


,support,itemsets,length
0,0.106357,(WHITE HANGING HEART T-LIGHT HOLDER),1
1,0.017321,(RED WOOLLY HOTTIE WHITE HEART.),1
2,0.017213,(KNITTED UNION FLAG HOT WATER BOTTLE),1
3,0.016026,(SET 7 BABUSHKA NESTING BOXES),1
4,0.013005,(CREAM CUPID HEARTS COAT HANGER),1


In [11]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
strong_rules = rules[(rules['confidence'] >= 0.6) & (rules['lift'] >= 1.2)]
strong_rules = strong_rules.sort_values(by='lift', ascending=False)
strong_rules.head()


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
889,(REGENCY TEA PLATE PINK),(REGENCY TEA PLATE GREEN ),0.012087,0.014569,0.010900,0.901786,61.895899,1.0,0.010724,10.033475,0.995881,0.691781,0.900334,0.824967
888,(REGENCY TEA PLATE GREEN ),(REGENCY TEA PLATE PINK),0.014569,0.012087,0.010900,0.748148,61.895899,1.0,0.010724,3.922595,0.998390,0.691781,0.745067,0.824967
594,(POPPY'S PLAYHOUSE LIVINGROOM ),"(POPPY'S PLAYHOUSE KITCHEN, POPPY'S PLAYHOUSE ...",0.013598,0.013706,0.010037,0.738095,53.851894,1.0,0.009850,3.765850,0.994960,0.581250,0.734456,0.735189
591,"(POPPY'S PLAYHOUSE KITCHEN, POPPY'S PLAYHOUSE ...",(POPPY'S PLAYHOUSE LIVINGROOM ),0.013706,0.013598,0.010037,0.732283,53.851894,1.0,0.009850,3.684501,0.995069,0.581250,0.728593,0.735189
885,(REGENCY MILK JUG PINK ),(REGENCY SUGAR BOWL GREEN),0.014677,0.014461,0.011116,0.757353,52.370391,1.0,0.010904,4.061613,0.995517,0.616766,0.753792,0.763005


In [13]:
def recommend_products(item, rules_df, top_n=5):
    matching_rules = rules_df[rules_df['antecedents'].apply(lambda x: item in x)]
    if matching_rules.empty:
        return f"No recommendations found for '{item}'."
    
    matching_rules = matching_rules.sort_values(by=['confidence', 'lift'], ascending=False)
    recommendations = matching_rules['consequents'].apply(lambda x: list(x)[0]).head(top_n).tolist()
    return recommendations


In [25]:
item_name = 'PINK REGENCY TEACUP AND SAUCER'
recommendations = recommend_products(item_name, strong_rules)
print(f"Recommended products for '{item_name}':", recommendations)

Recommended products for 'PINK REGENCY TEACUP AND SAUCER': ['GREEN REGENCY TEACUP AND SAUCER', 'GREEN REGENCY TEACUP AND SAUCER', 'ROSES REGENCY TEACUP AND SAUCER ', 'GREEN REGENCY TEACUP AND SAUCER', 'ROSES REGENCY TEACUP AND SAUCER ']


In [27]:
recommendation_table = get_recommendation_table(item_name, strong_rules)
display(recommendation_table)

,antecedents,consequents,support,confidence,lift
642,"(REGENCY CAKESTAND 3 TIER, ROSES REGENCY TEACU...",(GREEN REGENCY TEACUP AND SAUCER),0.012897,0.901887,24.187795
655,"(PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY...",(GREEN REGENCY TEACUP AND SAUCER),0.021045,0.894495,23.989564
640,"(REGENCY CAKESTAND 3 TIER, GREEN REGENCY TEACU...",(ROSES REGENCY TEACUP AND SAUCER ),0.012897,0.881919,20.873205
629,"(REGENCY CAKESTAND 3 TIER, PINK REGENCY TEACUP...",(GREEN REGENCY TEACUP AND SAUCER),0.014623,0.877023,23.520961
634,"(REGENCY CAKESTAND 3 TIER, PINK REGENCY TEACUP...",(ROSES REGENCY TEACUP AND SAUCER ),0.014300,0.857605,20.297751
654,"(PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY...",(ROSES REGENCY TEACUP AND SAUCER ),0.021045,0.847826,20.066300
622,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.024822,0.827338,22.188466
626,(PINK REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER ),0.023527,0.784173,18.559754
645,"(REGENCY CAKESTAND 3 TIER, PINK REGENCY TEACUP...","(GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...",0.012897,0.773463,26.495032
657,(PINK REGENCY TEACUP AND SAUCER),"(GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...",0.021045,0.701439,24.027846
